<a href="https://colab.research.google.com/github/adharshkamath/syncode/blob/popl/syncode_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install syncode

In [ ]:
# !pip install transformers==4.40.0

In [ ]:
grammar = """
start: operand operator operand

operand: FLOAT
operator: ADD | SUB | MUL | DIV

DIGIT: "0".."9"
INT: DIGIT+
SIGNED_INT: ["+"|"-"] INT
DECIMAL: INT "." INT? | "." INT

_EXP: ("e"|"E") SIGNED_INT
_FLOAT: INT _EXP | DECIMAL _EXP?
FLOAT: ("+"|"-")? _FLOAT

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

%import common.WS
%ignore WS
"""

In [ ]:
from syncode import Syncode
import warnings
warnings.filterwarnings('ignore')

model_name = "microsoft/phi-2"

# Load the Syncode augmented model
# clear syn_llm if it exists
try:
    del syn_llm
except:
    pass

syn_llm = Syncode(model=model_name, grammar=grammar, max_new_tokens=100, mode='grammar_strict')
syn_llm.model.tokenizer.chat_template = 'QA'

In [ ]:
prompt = "Given a question about two numbers " \
+ "and an operator (addition, multiplication, subtraction or division), output the corresponding mathematical expression. Below are some examples, for reference. " + \
"\nExample:\n"
few_shot_examples = [
    "<|question|> Student: What is 45.1 plus 23.54? <|question_end|>\nTutor: 45.1 + 22.2",
    "<|question|> Student: What is 120.4 divided by 4.0? <|question_end|>\nTutor: 120.4 / 4.0",
    "<|question|> Student: What is 327. multiplied by 11.0? <|question_end|>\nTutor: 327.0 * 11.0"
]

In [ ]:
def run_syncode(textual_query):
    final_prompt = prompt + "\n".join(few_shot_examples) + \
    "\n-----\nNow do the same for the following question:\n<|question|> Student: " \
    + textual_query + "<|question_end|>\nTutor:"
    print("Prompt: ")
    print(final_prompt)
    syncode_response = syn_llm.infer(final_prompt)
    print("LLM response: ")
    print(syncode_response) # raw response from the LLM

    if syn_llm.grammar_decoder is None:
        return syncode_response # for debugging, return raw resp if not grammar mode

    # getting the parsed tokens from syncode's state
    parsed_tokens = syn_llm.grammar_decoder.grammar_engine.inc_parser.parsed_lexer_tokens
    operands = [float(x.value) for x in parsed_tokens if x.type == 'FLOAT']
    operator = [x.value for x in parsed_tokens if x.type in ['SUB', 'ADD', 'MUL', 'DIV']][0]
    print("Parsed operands: ", operands, "and operator: ", operator)

    # evaluating the expression using python's eval()
    expression = str(operands[0]) + " " + operator + " " + str(operands[1])
    result = eval(expression)
    print("Result: ", result)
    return operands, operator, result

In [ ]:
response = run_syncode("What is 2032.1 subtracted by 21.2?")
print(response)